# AOED / PanTranscriptome -- download + QC + quantificacao (Colab)

Este notebook substitui a etapa de download+quantificacao que antes rodava localmente via WSL2 + Nextflow (muito mais lenta de rede neste computador). Aqui fazemos tudo com ferramentas diretas (sem Docker, sem Nextflow -- o Colab nao suporta Docker):

`wget` (baixa do ENA) -> `fastqc` (QC bruto) -> `fastp` (trimming) -> `fastqc` (QC pos-trim) -> `salmon quant` (quantificacao, decoy-aware, igual ao que ja validamos localmente para DRR786485) -> agregacao a nivel de gene via `t2g.tsv`.

Ao final de cada amostra, so os **resultados de analise** ficam salvos no seu Drive (FastQC, fastp, `quant.sf`, contagens por gene) -- os FASTQ brutos e trimados sao sempre apagados do disco local do Colab, nunca vao pro Drive.

## Referencia no Drive

Voce ja subiu a pasta `database/FungiDB_v71/` inteira (com os nomes originais) para `MyDrive/PanTranscriptome/reference/` -- o notebook acha o genoma e o transcritoma sozinho por padrao de nome, nao precisa renomear nada.

Falta so mais um arquivo, que **nao** faz parte dessa pasta:

| Arquivo local | Onde colocar no Drive |
|---|---|
| `reference/transcripts/t2g.tsv` | `MyDrive/PanTranscriptome/reference/t2g.tsv` |

(O `annotation.gff` que ja esta la dentro nao e usado -- o indice do Salmon so precisa de genoma+transcritoma, e o `t2g.tsv` ja tem o mapeamento transcrito->gene pronto.)

## Como usar

1. Rode as celulas em ordem, ate a de instalacao do `condacolab` -- ela **reinicia o runtime automaticamente** (e normal, e a unica forma do condacolab funcionar). Depois do restart, so continue rodando da celula seguinte em diante (nao precisa rodar o mount do Drive nem o condacolab de novo).
2. Com `TEST_MODE = True` (padrao), ele processa so a primeira amostra da lista (`DRR059464`) como teste rapido.
3. Depois de confirmar que deu tudo certo, mude `TEST_MODE` para `False` na celula de configuracao, rode ela de novo e rode a celula final (driver) de novo -- ai ele processa a lista inteira.
4. Pode interromper a qualquer momento (Runtime > Interrupt execution) e rodar a celula final de novo depois -- amostras ja concluidas (com marcador `DONE`) sao puladas, e qualquer amostra que ficou pela metade e apagada e refeita do zero.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Instala o condacolab (unica forma pratica de ter conda/mamba no Colab).
# ATENCAO: isso reinicia o runtime sozinho -- e esperado, nao e erro.
# Depois do restart, so continue rodando a partir da proxima celula.
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# Rode esta celula DEPOIS do restart automatico da celula anterior.
#
# NAO instalamos salmon no ambiente base do condacolab: ele fixa
# python=3.13, e TODA build 3.13.x exige libzlib >=1.3.x, enquanto
# salmon=1.10.1 exige libzlib <1.3.0a0 -- incompativel de verdade, sem
# solucao dentro do ambiente base (confirmado 2026-09-21: o conda so
# "resolvia" isso removendo o proprio conda/mamba/python do ambiente).
# salmon=1.10.3 (versao mais nova, sem esse conflito de zlib) tambem nao
# serve -- crasha com SIGSEGV nesta CPU do Colab mesmo sozinho.
#
# Fix: ambiente conda separado com python=3.11 (sem o pin do condacolab),
# onde salmon=1.10.1 resolve sem conflito -- e essa build ja provou que
# NAO da SIGSEGV nesta CPU (chegou a imprimir a versao com sucesso antes
# do ambiente base quebrar). fastqc/fastp/pandas/requests vao no mesmo
# ambiente.
!conda create -y -n transcriptome -c conda-forge -c bioconda python=3.11 fastqc fastp pandas requests
!conda install -y -n transcriptome -c bioconda salmon=1.10.1

import os

TRANSCRIPTOME_ENV_BIN = "/usr/local/envs/transcriptome/bin"
if not os.path.isdir(TRANSCRIPTOME_ENV_BIN):
    raise RuntimeError(
        f"Nao achei {TRANSCRIPTOME_ENV_BIN} -- o ambiente 'transcriptome' nao foi criado como esperado."
    )
os.environ["PATH"] = TRANSCRIPTOME_ENV_BIN + ":" + os.environ["PATH"]

!salmon --version
!fastqc --version
!fastp --version

In [ ]:
import os
import glob

TEST_MODE = True  # Troque para False depois de validar com a 1a amostra

DRIVE_ROOT = "/content/drive/MyDrive/PanTranscriptome"
REFERENCE_DIR = f"{DRIVE_ROOT}/reference"
INDEX_DIR = f"{DRIVE_ROOT}/salmon_index"
RESULTS_DIR = f"{DRIVE_ROOT}/results"
LOGS_DIR = f"{DRIVE_ROOT}/logs"
PROGRESS_LOG = f"{LOGS_DIR}/progress.csv"


def find_one(pattern, what):
    matches = glob.glob(os.path.join(REFERENCE_DIR, pattern))
    if len(matches) == 0:
        raise FileNotFoundError(
            f"Nao achei {what} em {REFERENCE_DIR} (padrao '{pattern}'). Suba o arquivo no Drive antes de continuar."
        )
    if len(matches) > 1:
        raise FileNotFoundError(
            f"Mais de um arquivo casou com {what} em {REFERENCE_DIR}: {matches} -- apague o que sobrar."
        )
    return matches[0]


# Nomes originais do FungiDB (a pasta inteira foi enviada como esta no repo
# local, sem renomear) -- descobertos por padrao em vez de nome fixo.
GENOME_FASTA = find_one("*Genome.fasta", "genoma")
TRANSCRIPT_FASTA = find_one("*AnnotatedTranscripts.fasta", "transcritoma")
T2G_FILE = f"{REFERENCE_DIR}/t2g.tsv"  # este NAO faz parte do FungiDB_v71 -- upload separado

WORKDIR = "/content/work"  # disco local rapido do Colab -- nunca o Drive

DOWNLOAD_TIMEOUT_SECONDS = 3600  # limite de 1h por amostra, so para o download

# Lista fixa de accessions a processar. Adicione novas aqui conforme forem
# aprovadas (ver atlas sample approve / TODO.md no repo local).
ACCESSIONS = [
    "DRR059464",
    "DRR059465",
    "DRR059466",
]

for d in (WORKDIR, RESULTS_DIR, LOGS_DIR):
    os.makedirs(d, exist_ok=True)

if not os.path.exists(T2G_FILE):
    raise FileNotFoundError(
        f"Faltando {T2G_FILE} -- suba reference/transcripts/t2g.tsv (do repo local) pro Drive antes de continuar."
    )

print("Config OK. TEST_MODE =", TEST_MODE)
print("Genoma:", GENOME_FASTA)
print("Transcritoma:", TRANSCRIPT_FASTA)
print("t2g:", T2G_FILE)

In [ ]:
import subprocess, requests, hashlib, shutil, datetime, csv, sys

sys.setrecursionlimit(10000)


def run(cmd):
    print("  $", " ".join(cmd))
    subprocess.run(cmd, check=True)


def ena_runinfo(acc):
    """Mesma API que o nf-core/fetchngs usa por baixo dos panos para resolver
    accession -> URLs de FASTQ + MD5 + single/paired-end."""
    url = (
        "https://www.ebi.ac.uk/ena/portal/api/filereport"
        f"?accession={acc}&result=read_run&fields=fastq_ftp,fastq_md5,fastq_bytes,library_layout&format=tsv"
    )
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    lines = resp.text.strip().splitlines()
    if len(lines) < 2:
        raise RuntimeError(f"ENA nao retornou runinfo para {acc}")
    header = lines[0].split("\t")
    values = lines[1].split("\t")
    info = dict(zip(header, values))
    if not info.get("fastq_ftp"):
        raise RuntimeError(f"ENA sem fastq_ftp para {acc} (pode nao estar publicamente disponivel ainda)")
    return {
        "urls": info["fastq_ftp"].split(";"),
        "md5s": info["fastq_md5"].split(";"),
        "layout": info.get("library_layout", "SINGLE"),
    }


def md5sum(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def download_fastqs(acc, dest_dir, timeout=DOWNLOAD_TIMEOUT_SECONDS):
    info = ena_runinfo(acc)
    local_paths = []
    for url, expected_md5 in zip(info["urls"], info["md5s"]):
        filename = os.path.basename(url)
        local_path = os.path.join(dest_dir, filename)
        subprocess.run(
            ["wget", "-t", "5", "-nv", "-c", "-T", "60", "-O", local_path, url],
            check=True, timeout=timeout,
        )
        actual_md5 = md5sum(local_path)
        if actual_md5 != expected_md5:
            raise RuntimeError(f"MD5 nao bate para {filename}: esperado {expected_md5}, obtido {actual_md5}")
        local_paths.append(local_path)
    return local_paths, info["layout"]


def process_reads(fastq_paths, layout, work_dir):
    fastqc_raw_dir = os.path.join(work_dir, "fastqc_raw")
    fastqc_trim_dir = os.path.join(work_dir, "fastqc_trimmed")
    os.makedirs(fastqc_raw_dir, exist_ok=True)
    os.makedirs(fastqc_trim_dir, exist_ok=True)

    run(["fastqc", "-q", "-o", fastqc_raw_dir] + fastq_paths)

    is_paired = layout.upper() == "PAIRED" and len(fastq_paths) == 2
    trimmed_paths = [p.replace(".fastq.gz", ".trimmed.fastq.gz") for p in fastq_paths]
    fastp_json = os.path.join(work_dir, "fastp.json")
    fastp_html = os.path.join(work_dir, "fastp.html")

    if is_paired:
        run(["fastp", "-i", fastq_paths[0], "-I", fastq_paths[1],
             "-o", trimmed_paths[0], "-O", trimmed_paths[1],
             "--json", fastp_json, "--html", fastp_html, "--thread", "4"])
    else:
        trimmed_paths = trimmed_paths[:1]
        run(["fastp", "-i", fastq_paths[0], "-o", trimmed_paths[0],
             "--json", fastp_json, "--html", fastp_html, "--thread", "4"])

    run(["fastqc", "-q", "-o", fastqc_trim_dir] + trimmed_paths)

    salmon_out = os.path.join(work_dir, "salmon")
    if is_paired:
        run(["salmon", "quant", "-i", INDEX_DIR, "-l", "A",
             "-1", trimmed_paths[0], "-2", trimmed_paths[1],
             "--validateMappings", "-p", "4", "-o", salmon_out])
    else:
        run(["salmon", "quant", "-i", INDEX_DIR, "-l", "A",
             "-r", trimmed_paths[0],
             "--validateMappings", "-p", "4", "-o", salmon_out])

    return fastqc_raw_dir, fastqc_trim_dir, fastp_json, fastp_html, salmon_out


def aggregate_to_gene(quant_sf_path, t2g_path, out_path):
    import pandas as pd
    quant = pd.read_csv(quant_sf_path, sep="\t")
    t2g = pd.read_csv(t2g_path, sep="\t")
    merged = quant.merge(t2g, left_on="Name", right_on="transcript_id", how="left")
    missing = merged["gene_id"].isna().sum()
    if missing:
        print(f"  AVISO: {missing} transcritos sem gene correspondente em t2g.tsv")
    gene_level = merged.groupby("gene_id", as_index=False)[["TPM", "NumReads"]].sum()
    gene_level.to_csv(out_path, sep="\t", index=False)
    return out_path


def log_event(acc, status, detail=""):
    is_new = not os.path.exists(PROGRESS_LOG)
    with open(PROGRESS_LOG, "a", newline="") as f:
        w = csv.writer(f)
        if is_new:
            w.writerow(["timestamp", "accession", "status", "detail"])
        w.writerow([datetime.datetime.utcnow().isoformat(), acc, status, detail])


def sample_result_dir(acc):
    return os.path.join(RESULTS_DIR, acc)


def is_done(acc):
    return os.path.exists(os.path.join(sample_result_dir(acc), "DONE"))


def cleanup_partial(acc):
    d = sample_result_dir(acc)
    if os.path.exists(d):
        print(f"  Removendo pasta incompleta de {acc} em {d}")
        shutil.rmtree(d)


print("Funcoes auxiliares carregadas.")

In [ ]:
def build_salmon_index():
    """Constroi o indice Salmon decoy-aware uma unica vez (fica cacheado no
    Drive em INDEX_DIR e e reaproveitado em toda sessao futura)."""
    if os.path.exists(os.path.join(INDEX_DIR, "info.json")):
        print("Indice Salmon ja existe em", INDEX_DIR, "-- pulando construcao.")
        return

    print("Construindo indice Salmon decoy-aware (so na primeira vez, leva alguns minutos)...")
    os.makedirs(WORKDIR, exist_ok=True)
    decoys_txt = os.path.join(WORKDIR, "decoys.txt")
    gentrome_fa = os.path.join(WORKDIR, "gentrome.fa.gz")

    run(["bash", "-c",
         f"grep '^>' {GENOME_FASTA} | sed 's/^>//' | cut -d' ' -f1 > {decoys_txt}"])
    run(["bash", "-c",
         f"cat {TRANSCRIPT_FASTA} {GENOME_FASTA} | gzip > {gentrome_fa}"])

    run(["salmon", "index", "-t", gentrome_fa, "-d", decoys_txt,
         "-i", INDEX_DIR, "-k", "31", "-p", "4"])


build_salmon_index()

In [ ]:
def process_sample(acc):
    print(f"\n=== {acc} ===")
    if is_done(acc):
        print(f"{acc}: ja concluido, pulando.")
        return

    # pasta existe mas sem marcador DONE -> tentativa anterior ficou pela metade
    cleanup_partial(acc)

    work_dir = os.path.join(WORKDIR, acc)
    if os.path.exists(work_dir):
        shutil.rmtree(work_dir)
    os.makedirs(work_dir, exist_ok=True)

    try:
        print(f"{acc}: baixando FASTQ...")
        fastq_paths, layout = download_fastqs(acc, work_dir)

        print(f"{acc}: QC + trimming + quantificacao Salmon...")
        fastqc_raw, fastqc_trim, fastp_json, fastp_html, salmon_out = process_reads(
            fastq_paths, layout, work_dir
        )

        quant_sf = os.path.join(salmon_out, "quant.sf")
        gene_quant_path = os.path.join(work_dir, "gene_quant.tsv")
        aggregate_to_gene(quant_sf, T2G_FILE, gene_quant_path)

        # Monta a pasta final so com o que vamos usar depois (sem reads brutos/trimados)
        final_dir = sample_result_dir(acc)
        os.makedirs(final_dir, exist_ok=True)
        shutil.copytree(fastqc_raw, os.path.join(final_dir, "fastqc_raw"))
        shutil.copytree(fastqc_trim, os.path.join(final_dir, "fastqc_trimmed"))
        shutil.copy(fastp_json, final_dir)
        shutil.copy(fastp_html, final_dir)
        shutil.copy(quant_sf, os.path.join(final_dir, "quant.sf"))
        shutil.copy(gene_quant_path, final_dir)

        with open(os.path.join(final_dir, "DONE"), "w") as f:
            f.write(datetime.datetime.utcnow().isoformat() + "\n")

        log_event(acc, "done")
        print(f"{acc}: OK, salvo em {final_dir}")

    except subprocess.TimeoutExpired:
        print(f"{acc}: download excedeu {DOWNLOAD_TIMEOUT_SECONDS}s, pulando para a proxima.")
        log_event(acc, "error", f"download timeout ({DOWNLOAD_TIMEOUT_SECONDS}s)")
        cleanup_partial(acc)
    except Exception as e:
        print(f"{acc}: ERRO -- {e}")
        log_event(acc, "error", str(e))
        cleanup_partial(acc)
    finally:
        if os.path.exists(work_dir):
            shutil.rmtree(work_dir)  # nunca deixa reads no disco local do Colab


def process_all(accessions, i=0):
    if i >= len(accessions):
        print("\nLote finalizado.")
        return
    process_sample(accessions[i])
    process_all(accessions, i + 1)


targets = ACCESSIONS[:1] if TEST_MODE else ACCESSIONS
print(f"Modo: {'TESTE (1 amostra)' if TEST_MODE else 'COMPLETO'} -- {targets}")
process_all(targets)

## Proximos passos

- Se o teste acima terminou com `DRR059464: OK`, va na celula de configuracao, mude `TEST_MODE` para `False`, rode aquela celula de novo e depois rode a celula do driver (a de cima) de novo -- ele vai pular `DRR059464` (ja tem `DONE`) e processar `DRR059465` e `DRR059466`.
- O log de progresso fica em `MyDrive/PanTranscriptome/logs/progress.csv` (uma linha por evento: `done` ou `error`).
- Os resultados finais de cada amostra ficam em `MyDrive/PanTranscriptome/results/<ACCESSION>/` -- e so essa pasta (`fastqc_raw/`, `fastqc_trimmed/`, `fastp.json`, `fastp.html`, `quant.sf`, `gene_quant.tsv`, `DONE`) que precisa vir de volta pro PC, nunca os FASTQ brutos.
- Para adicionar novas amostras no futuro, so acrescente o accession na lista `ACCESSIONS` na celula de configuracao.